# 9.6 - Eval, Merge & Deploy

**Module 9 - Fine-Tuning** | Netsetos GenAI Engineering

This notebook is the Module 9 closer: it takes a fine-tuned adapter and ships it. Eight small, deterministic Python cells stand in for the real production tools so you can run the whole pipeline end to end - prove the fine-tune beats the base (held-out A/B + LLM-judge), read a benchmark portfolio, defend an LLM judge against its biases, gate on safety regression, merge the adapter (linear vs TIES/DARE-TIES), quantize to the target, pick a deploy target, and plan an India-specific path.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This lesson leans on seeded, hand-written values instead of random draws so every cell prints the exact same numbers each run. There is nothing to install and no model to load - the whole notebook is pure Python.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-line reproducibility note, not executable logic - it flags that all the figures downstream are fixed by design so your output matches the walkthroughs exactly.

**How the code works - step by step**
- A single comment marking that the lesson uses seeded / fixed values throughout, so results are deterministic.

**In one line:** no imports, no install - just a promise that every cell prints the same numbers every time.

**What you'll see:** (no output - environment prep)

## 1 - Prove it beats the base: held-out A/B

The first question is never "is the model good?" - it is "is it *better than what I already had?*" So you run the same unseen questions through both the base and the fine-tuned model, score them identically, and look only at the **delta**. A positive delta is your ship signal; a beautiful training-loss curve is not.

In [ ]:
# A/B EVAL - base vs fine-tuned on held-out cases (deterministic rule-judge stands in for the LLM).
def judge(response, expected_keywords):
    hits = sum(1 for k in expected_keywords if k.lower() in response.lower())
    return round(hits / len(expected_keywords), 3)

cases = [
    {"kw": ["7 days", "50%", "30"], "base": "Please check the website.",
     "ft": "Full refund within 7 days, 50% from 8-30 days, none after 30."},
    {"kw": ["14999", "lifetime"],   "base": "I am not sure of the price.",
     "ft": "It is 14999 rupees with lifetime access."},
    {"kw": ["python", "math"],      "base": "Various skills are needed.",
     "ft": "Python basics and high-school math."},
]
def eval_model(which):
    return round(sum(judge(c[which], c["kw"]) for c in cases) / len(cases), 3)

base, ft = eval_model("base"), eval_model("ft")
print(f"  base       score {base}")
print(f"  fine-tuned score {ft}   delta {ft-base:+.3f}")
print(f"  verdict: {'SHIP - fine-tune beats base' if ft > base else 'DO NOT SHIP - no gain over base'}")

# Output:
#   base       score 0.0
#   fine-tuned score 1.0   delta +1.000
#   verdict: SHIP - fine-tune beats base

**Explanation**

This is an evaluation harness, not a model call. A deterministic rule-based `judge` stands in for a real LLM judge so the A/B runs the same way every time - it scores a response by how many expected facts it contains. The point is the structure: identical cases, both models, identical scoring, compare the difference.

**How the code works - step by step**
- **`judge`** - counts how many of a case's `expected_keywords` appear in a response and returns the fraction (0-1); a cheap, deterministic proxy for an LLM judge.
- **`cases`** - three held-out support questions, each with expected keywords plus a weak `base` answer and a fact-rich `ft` (fine-tuned) answer.
- **`eval_model`** - runs all cases through one model (`"base"` or `"ft"`) and averages the per-case scores.
- **delta + verdict** - subtracts base from fine-tuned; a positive delta prints SHIP, otherwise DO NOT SHIP.

**In one line:** same cases through both models, identical scoring, ship only on a positive delta.

**What you'll see:** base scores 0.0, fine-tuned scores 1.0 for a delta of +1.000, and the verdict prints "SHIP - fine-tune beats base".

## 2 - The benchmark portfolio: why MMLU alone lies

Public benchmarks let you compare against known baselines - but the famous ones are **saturated**: strong models all cluster near the top, so the benchmark stops telling them apart. This cell encodes the rule for keeping a benchmark honest: it must be unsaturated *and* actually separate the two models.

In [ ]:
# BENCHMARK PORTFOLIO - a benchmark only helps if it is NOT saturated AND separates the two models.
bench = {
    "MMLU":         {"a": 0.88, "b": 0.89, "saturated": True},
    "MMLU-Pro":     {"a": 0.72, "b": 0.79, "saturated": False},
    "GPQA-Diamond": {"a": 0.41, "b": 0.55, "saturated": False},
    "IFEval":       {"a": 0.68, "b": 0.83, "saturated": False},
}
print("  benchmark      A     B    gap   discriminates?")
for name, d in bench.items():
    gap = abs(d["a"] - d["b"])
    ok = (not d["saturated"]) and gap >= 0.03
    tag = "yes" if ok else ("SATURATED" if d["saturated"] else "too close")
    print(f"  {name:13s} {d['a']:.2f}  {d['b']:.2f}  {gap:.2f}   {tag}")

# Output:
#   benchmark      A     B    gap   discriminates?
#   MMLU          0.88  0.89  0.01   SATURATED
#   MMLU-Pro      0.72  0.79  0.07   yes
#   GPQA-Diamond  0.41  0.55  0.14   yes
#   IFEval        0.68  0.83  0.15   yes

**Explanation**

A tiny scoring filter over a benchmark table, not a real evaluation run. It applies two gates to each benchmark - not saturated, and a meaningful gap between models A and B - to decide whether that benchmark discriminates. It makes concrete why you report a portfolio (MMLU-Pro, GPQA, IFEval) instead of a single number.

**How the code works - step by step**
- **`bench`** - four benchmarks with scores for model `a`, model `b`, and a `saturated` flag (MMLU is flagged saturated).
- **loop over `bench`** - computes `gap = |a - b|` for each benchmark.
- **`ok` test** - a benchmark discriminates only if it is NOT saturated AND the gap is at least 0.03.
- **`tag`** - prints `yes`, `SATURATED`, or `too close` per benchmark.

**In one line:** a benchmark is useful only when it isn't maxed out AND still opens a real gap between the models.

**What you'll see:** A table where MMLU (gap 0.01) prints SATURATED, while MMLU-Pro (0.07), GPQA-Diamond (0.14), and IFEval (0.15) all print "yes" - they still discriminate.

## 3 - LLM-as-judge: cheap, fast, and biased

An LLM judge agrees with human preference ~80-90% of the time at a tiny fraction of the cost - which is why it is the default for open-ended scoring. But it carries known biases: position (favors whichever answer is shown first), verbosity, and self-enhancement. This cell shows the two cheapest defenses.

In [ ]:
# LLM-AS-JUDGE DEFENSES - position swap (accept only if consistent) + jury majority.
def biased_judge(favors_first=True):
    return "first" if favors_first else "second"   # a judge that leans to slot 1

def swap_consistent(favors_first):
    v1 = biased_judge(favors_first)                # A in slot 1
    v2 = biased_judge(favors_first)                # A in slot 2 -> "first" now means B
    win1 = "A" if v1 == "first" else "B"
    win2 = "B" if v2 == "first" else "A"
    return win1 if win1 == win2 else "TIE (inconsistent -> position bias detected)"

from collections import Counter
def jury(votes):
    top, n = Counter(votes).most_common(1)[0]
    return top if n > len(votes)/2 else "ABSTAIN (no majority)"

print(f"  slot-1-biased judge, swapped -> {swap_consistent(True)}")
print(f"  jury [A, A, B]   -> {jury(['A','A','B'])}")
print(f"  jury [A, B, TIE] -> {jury(['A','B','TIE'])}")

# Output:
#   slot-1-biased judge, swapped -> TIE (inconsistent -> position bias detected)
#   jury [A, A, B]   -> A
#   jury [A, B, TIE] -> ABSTAIN (no majority)

**Explanation**

A pair of defense mechanisms modeled with a deliberately biased stub judge - no API call. `swap_consistent` runs the judge in both orderings and only trusts a verdict that survives the flip; `jury` takes a majority vote across judge families and abstains when there's no majority. The bias is exposed, not hidden.

**How the code works - step by step**
- **`biased_judge`** - a stub that always leans toward one slot, standing in for a position-biased judge.
- **`swap_consistent`** - judges answer A in slot 1, then in slot 2, and re-maps each verdict back to A/B; if the two disagree it returns TIE, surfacing the position bias.
- **`jury`** - uses `Counter` to take a majority vote over a list of judge verdicts, returning ABSTAIN when no option clears half.
- **three prints** - the swapped biased judge, a clear-majority jury, and a no-majority jury.

**In one line:** swap the order to catch position bias, then vote a jury and abstain when it splits.

**What you'll see:** The slot-1-biased judge run swapped returns "TIE (inconsistent -> position bias detected)"; jury [A,A,B] returns A; jury [A,B,TIE] returns "ABSTAIN (no majority)".

## 4 - The safety gate: fine-tuning can break alignment

The finding that surprises everyone: fine-tuning on even *benign*, on-task data can silently strip a model's safety refusals (Qi et al. 2023). So the last gate before deploy is not accuracy - it is safety regression (does it still refuse?) plus catastrophic forgetting (perplexity on held-out text). This gate is a hard blocker.

In [ ]:
# SAFETY GATE - block the deploy on a safety-refusal drop OR catastrophic forgetting.
def deploy_gate(base_refusal, ft_refusal, base_ppl, ft_ppl):
    reasons = []
    if ft_refusal < 0.95:
        reasons.append(f"SAFETY: refusal {ft_refusal:.0%} < 95% (base {base_refusal:.0%}) - alignment degraded")
    if ft_ppl > base_ppl * 1.10:
        reasons.append(f"FORGETTING: perplexity {ft_ppl:.1f} > 1.1x base {base_ppl:.1f}")
    return ("BLOCK", reasons) if reasons else ("PASS", ["within thresholds"])

for label, args in [("benign fine-tune", (1.00, 0.62, 8.2, 8.4)),
                    ("safe fine-tune",   (1.00, 0.98, 8.2, 8.5))]:
    verdict, reasons = deploy_gate(*args)
    print(f"  {label:18s} -> {verdict}")
    for r in reasons: print(f"      - {r}")

# Output:
#   benign fine-tune   -> BLOCK
#       - SAFETY: refusal 62% < 95% (base 100%) - alignment degraded
#   safe fine-tune     -> PASS
#       - within thresholds

**Explanation**

A pass/block decision function, not a safety test itself - it encodes the two thresholds a fine-tune must clear. It checks the refusal rate against a 95% floor and perplexity against a 1.1x-of-base ceiling, and returns BLOCK with reasons if either fails. Accuracy is irrelevant here; failing this gate stops the deploy outright.

**How the code works - step by step**
- **`deploy_gate`** - takes base/fine-tuned refusal rates and base/fine-tuned perplexities; collects a reason if refusal drops below 95% or perplexity rises above 1.1x base.
- **return** - `BLOCK` with the failing reasons, or `PASS` when both thresholds hold.
- **loop** - runs two scenarios: a "benign fine-tune" whose refusals collapsed, and a "safe fine-tune" that held its line.

**In one line:** refusals must stay high AND perplexity must barely move, or the deploy is blocked.

**What you'll see:** The benign fine-tune prints BLOCK with "SAFETY: refusal 62% < 95%"; the safe fine-tune prints PASS "within thresholds".

## 5 - Merge the adapter: linear vs TIES vs DARE-TIES

Once a model clears eval and safety, you fold the adapter in - or blend whole fine-tuned models. The hard part is *conflicting* updates: two fine-tunes that push the same weight in opposite directions. This cell treats each fine-tune as a task vector (a delta vs the base) and merges four ways so you can see interference resolve.

In [ ]:
# MERGE MATH - two fine-tunes as task vectors (deltas vs base). Merge 4 ways.
import math
t1 = [ 0.9, -0.2,  0.5,  0.05, -0.7]      # support-bot delta
t2 = [ 0.8,  0.3, -0.6,  0.04,  0.7]      # coding-bot delta

def linear(a, b, w=0.5):
    return [round(w*x + (1-w)*y, 3) for x, y in zip(a, b)]

def ties(a, b, keep=0.6):
    def trim(v):                              # 1) keep the top-|magnitude| fraction
        k = max(1, int(len(v)*keep)); thr = sorted((abs(x) for x in v), reverse=True)[k-1]
        return [x if abs(x) >= thr else 0.0 for x in v]
    ta, tb = trim(a), trim(b); out = []
    for x, y in zip(ta, tb):
        s = 1 if x + y >= 0 else -1           # 2) ELECT a sign
        vals = [v for v in (x, y) if v != 0 and (v > 0) == (s > 0)]   # 3) keep only agreeing deltas
        out.append(round(sum(vals)/len(vals), 3) if vals else 0.0)
    return out

def dare_ties(a, b, drop=0.5):
    def dare(v):                              # DARE: drop deltas, rescale survivors by 1/(1-drop)
        return [0.0 if i % 2 == 0 else round(x/(1-drop), 3) for i, x in enumerate(v)]
    return ties(dare(a), dare(b))             # fixed alternating mask here for reproducibility

print(f"  t1 (support): {t1}")
print(f"  t2 (coding):  {t2}")
print(f"  linear 50/50: {linear(t1, t2)}   <- position 3 muddied to -0.05 (interference)")
print(f"  TIES:         {ties(t1, t2)}   <- sign-elected: position 3 kept -0.6, no muddy average")
print(f"  DARE-TIES:    {dare_ties(t1, t2)}   <- dropped+rescaled deltas, then TIES")
print("  DARE-TIES is strongest for SFT+DPO merges; SLERP DEGRADES aligned models")

# Output:
#   t1 (support): [0.9, -0.2, 0.5, 0.05, -0.7]
#   t2 (coding):  [0.8, 0.3, -0.6, 0.04, 0.7]
#   linear 50/50: [0.85, 0.05, -0.05, 0.045, 0.0]   <- position 3 muddied to -0.05 (interference)
#   TIES:         [0.85, 0.0, -0.6, 0.0, 0.7]   <- sign-elected: position 3 kept -0.6, no muddy average
#   DARE-TIES:    [0.0, 0.6, 0.0, 0.09, 0.0]   <- dropped+rescaled deltas, then TIES

**Explanation**

A from-scratch numerical model of merge algorithms operating on two small delta vectors - no PEFT or mergekit here, just the math. `linear` averages (and muddies conflicts), `ties` trims small deltas then elects a sign per weight to keep only agreeing values, and `dare_ties` adds a drop-and-rescale step before TIES. It shows *why* TIES-family merges beat plain averaging.

**How the code works - step by step**
- **`t1`, `t2`** - two task vectors (support-bot and coding-bot deltas) that conflict at position 3 (+0.5 vs -0.6).
- **`linear`** - weighted average of the two vectors; opposite signs cancel into a muddy near-zero.
- **`ties`** - `trim` keeps the top-magnitude fraction, then per weight it *elects a sign* and averages only the deltas that agree with it.
- **`dare_ties`** - `dare` drops half the deltas (fixed alternating mask for reproducibility) and rescales survivors by 1/(1-drop), then runs TIES.
- **four prints** - the two inputs plus linear, TIES, and DARE-TIES results, annotated at the conflicting position.

**In one line:** linear averaging muddies conflicts; TIES sign-elects to commit a direction; DARE-TIES drops-and-rescales first.

**What you'll see:** Position 3 muddies to -0.05 under linear but stays a committed -0.6 under TIES; DARE-TIES prints its dropped-and-rescaled vector [0.0, 0.6, 0.0, 0.09, 0.0].

## 5b - The real merge one-liners: PEFT + mergekit

The previous cell showed the merge *math*; this one shows the actual commands you'd run in production - folding a LoRA adapter with PEFT, or blending whole models with a mergekit YAML config.

In [ ]:
# THE REAL MERGES (illustrative). LoRA merge = PEFT; whole-model merge = mergekit.
merge_lora = """
from peft import AutoPeftModelForCausalLM
model = AutoPeftModelForCausalLM.from_pretrained("./netsetos-lora", device_map="auto")
merged = model.merge_and_unload()          # fold B*A into W0 -> standalone model
merged.save_pretrained("./netsetos-merged")
"""
mergekit_yaml = """
# pip install mergekit ; mergekit-yaml config.yaml ./out
models:
  - model: netsetos/support-v1
  - model: netsetos/coding-v1
merge_method: dare_ties          # best for SFT+DPO-aligned models (SLERP degrades them)
base_model: google/gemma-3-4b-it # pick the current base
dtype: bfloat16
"""
print("LoRA merge -> merge_and_unload; whole-model -> mergekit (dare_ties). Re-run eval on the merged model.")

# Output:
# LoRA merge -> merge_and_unload; whole-model -> mergekit (dare_ties). Re-run eval on the merged model.

**Explanation**

A reference cell, not a computation - it holds the real merge recipes as strings and prints a reminder. The LoRA path uses PEFT's `merge_and_unload`; the whole-model path uses a mergekit `dare_ties` config. Note the key caveat carried from earlier: re-run eval + safety on the merged result.

**How the code works - step by step**
- **`merge_lora`** - a string showing the PEFT flow: load the adapter with `AutoPeftModelForCausalLM`, call `merge_and_unload()` to fold B*A into W0, then `save_pretrained` a standalone model.
- **`mergekit_yaml`** - a string config listing two source models, `merge_method: dare_ties`, a base model, and bf16 dtype (with the note that SLERP degrades aligned models).
- **print** - a one-line reminder to re-run eval on the merged model.

**In one line:** `merge_and_unload` for a LoRA adapter, mergekit `dare_ties` for whole models - then re-evaluate.

**What you'll see:** One printed line: "LoRA merge -> merge_and_unload; whole-model -> mergekit (dare_ties). Re-run eval on the merged model." The recipe strings are defined but not executed.

## 6 - Quantize for the target: GGUF, AWQ, FP8

A merged fp16 7B is ~14GB - too big for a laptop, wasteful on a GPU. Quantization shrinks it, but the right format depends on *where* it runs, and the subtle part is that a quantization algorithm is only as fast as its kernel. This cell lays out the size/quality/target tradeoffs side by side.

In [ ]:
# QUANTIZE FOR THE TARGET - size, quality, and where each format belongs (7B model).
def quant(fmt, B):
    table = {
        "fp16":       (B*2.0,  1.00, "baseline"),
        "GGUF Q6_K":  (B*0.79, 0.98, "local, quality-sensitive"),
        "GGUF Q4_K_M":(B*0.63, 0.94, "local / Ollama, VRAM-tight"),
        "AWQ+Marlin": (B*0.6,  0.96, "GPU throughput (needs the Marlin kernel)"),
        "FP8 W8A8":   (B*1.0,  0.99, "H100/H200 only"),
    }
    return table[fmt]

print(f"  {'format':12s} {'VRAM':>7} {'quality':>8}   note")
for f in ("fp16", "GGUF Q6_K", "GGUF Q4_K_M", "AWQ+Marlin", "FP8 W8A8"):
    gb, q, note = quant(f, 7)
    print(f"  {f:12s} {gb:6.1f}GB {q*100:6.0f}%   {note}")
print("  kernels > algorithms: AWQ WITHOUT Marlin can be slower than fp16; AutoAWQ archived -> llm-compressor")

# Output:
#   format          VRAM  quality   note
#   fp16           14.0GB    100%   baseline
#   GGUF Q6_K       5.5GB     98%   local, quality-sensitive
#   GGUF Q4_K_M     4.4GB     94%   local / Ollama, VRAM-tight
#   AWQ+Marlin      4.2GB     96%   GPU throughput (needs the Marlin kernel)
#   FP8 W8A8        7.0GB     99%   H100/H200 only

**Explanation**

A lookup-table printer, not a real quantizer - it maps each format to an approximate VRAM footprint, a quality-retention percentage, and where it belongs. It makes the deployment-format decision concrete and flags the counter-intuitive rule that kernels matter more than the algorithm.

**How the code works - step by step**
- **`quant`** - given a format and model size B (billions of params), returns (VRAM GB, quality fraction, note) from a small table.
- **`table`** - fp16 baseline, GGUF Q6_K / Q4_K_M for local, AWQ+Marlin for GPU throughput, FP8 W8A8 for H100/H200.
- **loop** - prints each format's VRAM, quality %, and target note for a 7B model.
- **closing print** - the reminder that AWQ without the Marlin kernel can be slower than fp16, and AutoAWQ is archived (use llm-compressor).

**In one line:** match the format to the target hardware, and remember the kernel decides speed, not the algorithm.

**What you'll see:** A table for a 7B model: fp16 14.0GB/100%, GGUF Q6_K 5.5GB/98%, Q4_K_M 4.4GB/94%, AWQ+Marlin 4.2GB/96%, FP8 7.0GB/99%, plus the kernels-beat-algorithms note.

## 7 - Deploy: merge vs multi-LoRA, Ollama vs vLLM

The deploy decision is small and mechanical: it turns on task count, traffic, and whether it runs at the edge. This cell encodes that decision tree so a set of scenarios maps cleanly to Ollama, vLLM, vLLM multi-LoRA, or managed serving.

In [ ]:
# DEPLOY DECISION - by task count, traffic, and whether it runs at the edge.
def deploy(n_tasks, traffic, edge):
    if edge:
        return ("merge -> GGUF -> Ollama", "single model, CPU-friendly, runs local/edge")
    if n_tasks == 1:
        return ("merge_and_unload -> vLLM", "one standalone model, zero adapter overhead")
    if n_tasks > 1 and traffic != "high":
        return ("vLLM multi-LoRA (--enable-lora)", f"one base + {n_tasks} adapters routed per request")
    return ("managed (HF scale-to-zero / Vertex) or dedicated vLLM", "bursty/high; ops in Modules 12-14")

for label, args in [("1 task, edge", (1,"low",True)), ("1 task, GPU", (1,"med",False)),
                    ("4 tasks, GPU", (4,"med",False)), ("bursty many", (6,"high",False))]:
    tgt, why = deploy(*args)
    print(f"  {label:14s} -> {tgt}")

# Output:
#   1 task, edge   -> merge -> GGUF -> Ollama
#   1 task, GPU    -> merge_and_unload -> vLLM
#   4 tasks, GPU   -> vLLM multi-LoRA (--enable-lora)
#   bursty many    -> managed (HF scale-to-zero / Vertex) or dedicated vLLM

**Explanation**

A decision-tree function, not a serving call. Given the number of tasks, a traffic level, and an edge flag, it returns the recommended deploy target and a one-line rationale. It's the shipping decision this lesson owns - the deep serving ops (SLOs, autoscaling, canary) are explicitly deferred to Modules 12-14.

**How the code works - step by step**
- **`deploy`** - branches: edge -> merge to GGUF on Ollama; single task on GPU -> `merge_and_unload` on vLLM; multiple tasks at non-high traffic -> vLLM multi-LoRA (one base + N adapters); otherwise -> managed (HF scale-to-zero / Vertex) or dedicated vLLM.
- **loop** - runs four scenarios (1 task edge, 1 task GPU, 4 tasks GPU, bursty many) and prints the chosen target for each.

**In one line:** edge->Ollama, one GPU task->vLLM, many tasks->multi-LoRA, bursty->managed.

**What you'll see:** Four lines mapping each scenario to its target: edge->Ollama, 1-task GPU->vLLM merge_and_unload, 4-tasks GPU->vLLM multi-LoRA, bursty->managed/dedicated vLLM.

## 8 - The India deploy path: Sarvam, IndiaAI, Bhashini, DPDP

For Indian deployments the stack is concrete, cheap, and sovereign: subsidized IndiaAI/E2E GPUs, Sarvam's Indic-multilingual open models, Bhashini for 22-language speech and translation, and DPDP data-residency that Indian clouds satisfy by default. This cell is a budget-and-language planner over that stack.

In [ ]:
# INDIA DEPLOY PLANNER - budget + languages -> infra + model + Bhashini.
def india(budget_inr_month, langs):
    extra = "Bhashini APIs (STT/TTS/translate, 22 languages)" if langs > 1 else "single-language endpoint"
    if budget_inr_month < 15000:
        return ("HF scale-to-zero (T4) or Jarvislabs spot", "Sarvam-1 / Gemma 3, GGUF+Ollama", extra)
    if budget_inr_month <= 40000:
        return ("E2E / Jarvislabs H100 (~Rs 225-249/hr, peak hours only)", "Sarvam-M (24B) AWQ on vLLM", extra)
    return ("IndiaAI Mission H100 (~Rs 65/hr, subsidized)", "Sarvam-M / larger open model, vLLM multi-LoRA", extra)

for b, l in [(9000, 1), (30000, 3), (120000, 5)]:
    infra, model, extra = india(b, l)
    print(f"  budget Rs {b:<7}/mo langs={l} -> {infra}")
    print(f"      model: {model}")
print("  DPDP data-residency: Indian clouds comply by default - the natural pick for regulated sectors")

# Output:
#   budget Rs 9000   /mo langs=1 -> HF scale-to-zero (T4) or Jarvislabs spot
#       model: Sarvam-1 / Gemma 3, GGUF+Ollama
#   budget Rs 30000  /mo langs=3 -> E2E / Jarvislabs H100 (~Rs 225-249/hr, peak hours only)
#       model: Sarvam-M (24B) AWQ on vLLM
#   budget Rs 120000 /mo langs=5 -> IndiaAI Mission H100 (~Rs 65/hr, subsidized)
#       model: Sarvam-M / larger open model, vLLM multi-LoRA

**Explanation**

A planner function, not a deployment - it maps a monthly rupee budget plus a language count to a recommended infra tier, model, and multilingual add-on. The rupee GPU rates are hard-coded and indicative (they move over time), so the cell is about the decision shape, not exact prices.

**How the code works - step by step**
- **`india`** - given `budget_inr_month` and `langs`, first sets `extra` to Bhashini APIs when more than one language is needed.
- **budget branches** - under Rs 15k -> HF scale-to-zero / Jarvislabs spot with a small Sarvam/Gemma model on Ollama; up to Rs 40k -> E2E/Jarvislabs H100 with Sarvam-M on vLLM; above -> subsidized IndiaAI H100 with a larger model on vLLM multi-LoRA.
- **loop** - runs three (budget, language) tiers and prints infra + model per tier.
- **closing print** - the DPDP data-residency note: Indian clouds comply by default.

**In one line:** budget picks the GPU tier, language count adds Bhashini, and DPDP residency comes free on Indian clouds.

**What you'll see:** Three budget tiers print their infra + model: Rs 9k single-language -> scale-to-zero/Jarvislabs with Sarvam-1/Gemma; Rs 30k/3-lang -> E2E H100 with Sarvam-M; Rs 120k/5-lang -> IndiaAI H100 with vLLM multi-LoRA, plus the DPDP note.

Across eight runnable cells you walked the entire ship-it pipeline: the training-loss curve proves nothing, so you run a held-out A/B against the base (1), read it through a non-saturated benchmark portfolio (2), score open answers with an LLM judge whose position/verbosity biases you swap-and-jury away (3), and refuse to ship anything that fails the safety-refusal + perplexity gate (4). Only then do you merge the adapter - with TIES/DARE-TIES to resolve conflicting deltas instead of muddying them (5) - quantize to match the deployment target where kernels beat algorithms (6), pick Ollama/vLLM/multi-LoRA/managed by task count and traffic (7), and reach for the cheap, sovereign India path when residency demands it (8). Every stand-in here maps to a real tool (PEFT, mergekit, llm-compressor, vLLM); Modules 12-14 pick up the production ops - SLOs, autoscaling, canary, drift, and cost - that live downstream of this notebook.